# DO NOT CHANGE
- this notebooks is for creating the charts for the final submission report
- is applies Equal Waterfilling on org_id=1, in January

In [ ]:
import pandas as pd
from modules.load_config import load_params_from_yaml

import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import base64

from modules.pf_calculations import print_opt_energy_data, apply_pf_schedule_to_mps, plot_stacked_gain_loss_sortable, tf_to_int, tf_to_hourly, tf_to_daily
from modules.visualisations import plot_profile_by_category,plot_distribution_comparison,plot_sorted_mps_comparison, plot_sorted_mps_full

from plotly.io import to_html
from IPython.display import display, HTML
import cvxpy as cp

from modules.optimization_algorithms.NashProductOptAlgo import NashProductOptAlgo
from modules.optimization_algorithms.XNashProductOptAlgo import XNashProductOptAlgo
from modules.optimization_algorithms.EqualWaterfillingOptAlgo import EqualWaterfillingOptAlgo

In [ ]:
image_filename = "../../img/icon_logo-1024x1024.png"  # local file path
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

logo = dict(
    source=f"data:image/png;base64,{encoded_image}",  # local file embedded as base64
    xref="paper", yref="paper",
    x=0.5, y=0.5,            # position: centre
    sizex=0.6, sizey=0.6,    # adjust size
    xanchor="center",
    yanchor="middle",
    opacity=0.2,             # transparency
    layer="below"            # place below the data layer
)

## Params

In [ ]:
# # PARMS
# changeable
org_id = 1
start_time = datetime(2025, 1, 7)
end_time = datetime(2025, 1, 28)

hourly_mode = "max"
daily_mode = "max"

tf_to_evaluate = "daily" # "15min" # "daily" # "hourly"

evaluate_on = (
    "C"  # "G" - plotting charts for Generators # "C" - plotting charts for Consumers
)


# CONFIG FILE
path_to_local_data = "../../config/"
notebooks_config_file = "notebooks.yaml"

# overwriting variables based on params
energy_direction_filter = evaluate_on
if evaluate_on == "G":
    feature = "cons_gen"
elif evaluate_on == "C":
    feature = "comm_cov"
else:
    raise ValueError(f"Unknown evaluate_on value: {evaluate_on}")

## Load data

In [ ]:
config_dict = load_params_from_yaml([f"{path_to_local_data}{notebooks_config_file}"])
path_to_local_data = config_dict["PATH_TO_LOCAL_DATA"]
input_filename = config_dict["SINGLE_MPS_OF_EEG"].format(org_id=org_id)

full_filepath_to_load = f"{path_to_local_data}{input_filename}"

In [ ]:
raw_eeg = pd.read_csv(f"{full_filepath_to_load}")
raw_eeg['time'] = pd.to_datetime(raw_eeg['time'], utc=True)

print(f"{raw_eeg.dtypes}")
print(f"len: {len(raw_eeg)}")
raw_eeg.head()

In [ ]:
eeg_selected_feat = raw_eeg[["time", "organization_id", "metering_point_id", "energy_direction", "wt_meas_cons", "comm_pot", "comm_cov", "wt_meas_gen", "wt_surp_gen"]].copy()
del raw_eeg
print(eeg_selected_feat.dtypes)
print(f"len: {len(eeg_selected_feat)}")
print(eeg_selected_feat.drop("metering_point_id", axis=1).describe())
eeg_selected_feat.head()

In [ ]:
work = eeg_selected_feat[
    (eeg_selected_feat["time"] > pd.Timestamp(start_time, tz='UTC')) &
    (eeg_selected_feat["time"] < pd.Timestamp(end_time, tz='UTC'))
].drop_duplicates().copy()
del eeg_selected_feat
# cons_gen: Consumed Generation, how much of the generated electricity was consumed within the EEG
work["cons_gen"] = work["wt_meas_gen"] - work["wt_surp_gen"]
work.sum(numeric_only=True)

In [ ]:
work[work["energy_direction"]=="G"].head()

## eda

In [ ]:
plot_sorted_mps_full(data=work[work["energy_direction"]==energy_direction_filter], feature=feature)

In [ ]:
temp_gens = work[work["energy_direction"]==energy_direction_filter].groupby("metering_point_id").sum(numeric_only=True).drop(columns=["wt_meas_cons", "comm_pot", "comm_cov"])
temp_gens["cons_gen"] = temp_gens["wt_meas_gen"] - temp_gens["wt_surp_gen"]
temp_gens["cons_gen_to_wt_meas_gen_ratio"] = temp_gens["cons_gen"] / temp_gens["wt_meas_gen"]
temp_gens

In [ ]:
# temp_gens = work[(work["energy_direction"]==energy_direction_filter) & (work["time"].isin(time_with_surplus.time))].groupby("mp_id").sum(numeric_only=True).drop(columns=["wt_meas_cons", "comm_pot", "comm_cov"])
# temp_gens["cons_gen"] = temp_gens["wt_meas_gen"] - temp_gens["wt_surp_gen"]
# temp_gens["cons_gen_to_wt_meas_gen_ratio"] = temp_gens["cons_gen"] / temp_gens["wt_meas_gen"]
# temp_gens
# random_value = np.random.choice(time_with_surplus["time"].unique())

# temp_gens = (
#     work[
#         (work["energy_direction"] == energy_direction_filter)
#         & (work["time"] == random_value)
#     ]
#     .groupby(["time", "mp_id"])
#     .sum(numeric_only=True)
#     .drop(columns=["wt_meas_cons", "comm_pot", "comm_cov"])
#     .reset_index()
# )
# temp_gens["cons_gen"] = temp_gens["wt_meas_gen"] - temp_gens["wt_surp_gen"]
# temp_gens["cons_gen_to_wt_meas_gen_ratio"] = (
#     temp_gens["cons_gen"] / temp_gens["wt_meas_gen"]
# )
# temp_gens
# plot_sorted_mps(data=temp_gens, feature="cons_gen")

### Waterfilling Opt for finding optimal pfs

In [ ]:
mp_counts_on_time = (
    work
    .groupby("time")["energy_direction"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"C": "count_C_mps", "G": "count_G_mps"})
)
sums_on_time = work.groupby(by="time").sum().reset_index().drop(columns=["metering_point_id", "energy_direction"]).rename(columns={"wt_meas_cons":"sum_wt_meas_cons", "comm_pot":"sum_comm_pot", "comm_cov":"sum_comm_cov", "wt_meas_gen":"sum_wt_meas_gen", "wt_surp_gen":"sum_wt_surp_gen", "cons_gen":"sum_cons_gen"})

agg_on_time = pd.merge(left=sums_on_time, right=mp_counts_on_time, on="time", how="outer")
agg_on_time.head()

In [ ]:
time_with_deficit = agg_on_time[agg_on_time["sum_wt_surp_gen"] <= 0]
time_with_surplus = agg_on_time[agg_on_time["sum_wt_surp_gen"] > 0]

print(f"{len(time_with_deficit)}/{len(agg_on_time)} ({(len(time_with_deficit)/len(agg_on_time))*100:.4}%) timestamps has deficit. only for consumers during deficit a pf is optimized")
print(f"{len(time_with_surplus)}/{len(agg_on_time)} ({(len(time_with_surplus)/len(agg_on_time))*100:.4}%) timestamps has surplus. only for generators during surplus a pf is optimized")
time_with_deficit.head()

#### Acutal optimization calculation

In [ ]:
solver = cp.SCS # cp.SCS # cp.ECOS
algo = EqualWaterfillingOptAlgo ()
tf_schedule_raw = algo.calculate_pfs(work)

In [ ]:
tf_schedule_int = tf_to_int(tf_schedule_raw)
tf_schedule_hourly = tf_to_hourly(tf_schedule_int, mode=hourly_mode)
tf_schedule_daily = tf_to_daily(tf_schedule_hourly, mode=daily_mode)
tf_schedule_int.head()

### Apply pf schedule to energy data

In [ ]:
applied_15min_int_pfs = apply_pf_schedule_to_mps(work, tf_schedule_int)
applied_hourly_int_pfs = apply_pf_schedule_to_mps(work, tf_schedule_hourly)
applied_daily_int_pfs = apply_pf_schedule_to_mps(work, tf_schedule_daily)

In [ ]:
print_opt_energy_data(applied_15min_int_pfs, title="15min - Quarter-hourly PF")

In [ ]:
print_opt_energy_data(applied_hourly_int_pfs, title="hourly - Hourly PF")

In [ ]:
print_opt_energy_data(applied_daily_int_pfs, title="daily - Daily PF")

In [ ]:
move_on_cons = applied_daily_int_pfs[(applied_daily_int_pfs["energy_direction"]=="C") & ((applied_daily_int_pfs["comm_cov"] - applied_daily_int_pfs["opt_comm_cov"]).abs() <= 5e-4) & (applied_daily_int_pfs["pf"]!=1)]
print(move_on_cons[["time", "metering_point_id", "wt_meas_cons", "comm_cov", "opt_wt_meas_cons", "opt_comm_cov", "pf"]].head())

move_on_gens = applied_15min_int_pfs[(applied_15min_int_pfs["energy_direction"]=="G") & ((applied_15min_int_pfs["cons_gen"] - applied_15min_int_pfs["opt_cons_gen"]).abs() <= 5e-4) & (applied_15min_int_pfs["pf"]!=1)]
print(move_on_gens[["time", "metering_point_id", "wt_meas_gen", "cons_gen", "opt_wt_meas_gen", "opt_cons_gen", "pf"]].head())

#### Summed up for 15min

In [ ]:
tf_map = {
    "15min": applied_15min_int_pfs,
    "hourly": applied_hourly_int_pfs,
    "daily": applied_daily_int_pfs,
}

try:
    temp_applied_pfs_ed = tf_map[tf_to_evaluate]
except KeyError:
    raise ValueError(f"Unknown tf_to_evaluate: {tf_to_evaluate}")


In [ ]:
check_full_calc_via_time = temp_applied_pfs_ed.groupby(by="time").sum()[
    [
        "wt_meas_cons",
        "comm_cov",
        "wt_meas_gen",
        "wt_surp_gen",
        "cons_gen",
        "opt_wt_meas_cons",
        "opt_comm_cov",
        "opt_wt_meas_gen",
        "opt_wt_surp_gen",
        "opt_cons_gen"
    ]
].add_prefix("sum_")
check_full_calc_via_time

#### Summed up for single MP over whole time horizon

In [ ]:
check_full_calcc_via_mpid = temp_applied_pfs_ed.groupby(by="metering_point_id").sum(numeric_only=True)[
    [
        "wt_meas_cons",
        "comm_cov",
        "wt_meas_gen",
        "wt_surp_gen",
        "cons_gen",
        "opt_wt_meas_cons",
        "opt_comm_cov",
        "opt_wt_meas_gen",
        "opt_wt_surp_gen",
        "opt_cons_gen",
        "comm_cov_delta",
        "cons_gen_delta"
    ]
].add_prefix("sum_")
check_full_calcc_via_mpid.sort_values(by="sum_comm_cov_delta", ascending=False)

#### Summed up for single MP over single timestamp

### Result evaluation

In [ ]:
# OPTIMIZATION CHANGES on single timestamp t / single 15min
random_timestamp_from_tf_opt = np.random.choice(time_with_surplus.time)
print(
    "Random timestamp from time where pf opt got applied:", random_timestamp_from_tf_opt
)
# random_timestamp_from_tf_opt = datetime.fromisoformat(
#     "2025-01-24 04:45:00+00:00"
# )
# 2) Filter dt by this value
single_time_filtered = temp_applied_pfs_ed[temp_applied_pfs_ed["time"] == random_timestamp_from_tf_opt]

check_full_calc_via_single_timestamp = (
    single_time_filtered[single_time_filtered["energy_direction"]==evaluate_on]
    .groupby(by="metering_point_id")
    .sum(numeric_only=True)[
        [
        "wt_meas_cons",
        "comm_cov",
        "wt_meas_gen",
        "wt_surp_gen",
        "cons_gen",
        "opt_wt_meas_cons",
        "opt_comm_cov",
        "opt_wt_meas_gen",
        "opt_wt_surp_gen",
        "opt_cons_gen",
        "comm_cov_delta",
        "cons_gen_delta"
    ]
    ]
).add_prefix("sum_")
print(check_full_calc_via_single_timestamp.sum(numeric_only=True))
check_full_calc_via_single_timestamp.sort_values(by=f"sum_{feature}_delta", ascending=False)
plot_stacked_gain_loss_sortable(check_full_calc_via_single_timestamp, f"sum_{feature}", f"sum_opt_{feature}")

In [ ]:
print(single_time_filtered[single_time_filtered["metering_point_id"].isin([12581, 12622])][["time", "organization_id", "metering_point_id", "wt_meas_cons", "comm_cov", "pf", "opt_wt_meas_cons", "opt_comm_cov"]].round(3))

In [ ]:
# OPTIMIZATION CHANGES on single timestamp t / single 15min
random_timestamp_from_tf_opt = np.random.choice(time_with_deficit.time.dt.date.unique())
print(
    "Random timestamp from time where pf opt got applied:", random_timestamp_from_tf_opt
)
# 2) Filter dt by this value
single_time_filtered = temp_applied_pfs_ed[
    (temp_applied_pfs_ed["time"].dt.date == random_timestamp_from_tf_opt)
]



In [ ]:
check_full_calc_via_single_timestamp = (
    single_time_filtered[single_time_filtered["energy_direction"]==evaluate_on]
    .groupby(by="metering_point_id")
    .sum(numeric_only=True)[
        [
        "wt_meas_cons",
        "comm_cov",
        "wt_meas_gen",
        "wt_surp_gen",
        "cons_gen",
        "opt_wt_meas_cons",
        "opt_comm_cov",
        "opt_wt_meas_gen",
        "opt_wt_surp_gen",
        "opt_cons_gen",
        "comm_cov_delta",
        "cons_gen_delta"
    ]
    ]
).add_prefix("sum_")
# print(check_full_calc_via_single_timestamp.sum(numeric_only=True))
check_full_calc_via_single_timestamp.sort_values(by=f"sum_{feature}_delta", ascending=False)
plot_stacked_gain_loss_sortable(check_full_calc_via_single_timestamp, f"sum_{feature}", f"sum_opt_{feature}")

In [ ]:
print(f"{single_time_filtered.time.min()} - {single_time_filtered.time.max()}")

In [ ]:
single_time_filtered = temp_applied_pfs_ed # [temp_applied_pfs_ed["time"].isin(time_with_deficit.time.unique())]
check_full_calc_via_single_timestamp = (
    single_time_filtered[single_time_filtered["energy_direction"]==evaluate_on]
    .groupby(by="metering_point_id")
    .sum(numeric_only=True)[
        [
        "wt_meas_cons",
        "comm_cov",
        "wt_meas_gen",
        "wt_surp_gen",
        "cons_gen",
        "opt_wt_meas_cons",
        "opt_comm_cov",
        "opt_wt_meas_gen",
        "opt_wt_surp_gen",
        "opt_cons_gen",
        "comm_cov_delta",
        "cons_gen_delta"
    ]
    ]
).add_prefix("sum_")
# print(check_full_calc_via_single_timestamp.sum(numeric_only=True))

check_full_calc_via_single_timestamp.sort_values(by=f"sum_{feature}_delta", ascending=False)
plot_stacked_gain_loss_sortable(check_full_calc_via_single_timestamp, f"sum_{feature}", f"sum_opt_{feature}")

In [ ]:
print(single_time_filtered[single_time_filtered["metering_point_id"].isin([12581, 12622])].groupby("metering_point_id").sum(numeric_only=True)[["wt_meas_cons", "comm_cov", "opt_wt_meas_cons", "opt_comm_cov"]].round(3))

In [ ]:
# TODO cahrt with 2 comparing histgrams, comparing between cc and cc*

In [ ]:
# OPTIMIZATION CHANGES on while time horizon
plot_stacked_gain_loss_sortable(sums_on_cons_mps, "comm_cov", "opt_comm_cov")

#### Explanation of 15min-Opt vs "Full Horizon"-Opt

In [ ]:
obj_ids_to_filter = [238, 134, 162, 25]

temp = applied_pfs[applied_pfs["energy_direction"] == "C"]
#temp = temp[temp["mp_id"].isin(obj_ids_to_filter)]
plot_profile_by_category(temp, energy_col_name='wt_meas_cons', agg_func_str='median', hue_col="metering_point_id", logo=logo)

- chart nash -> no change
perspective: single perspective: no change in comm_cov, but tf was set

-> perspective eeg sums -> no change in comm_cov, but transfer of cons


- pipeline sketch
- pipeline draw io